# Aula 2: funções, comprehensions e erros (resolvido)

Na aula 1 vimos tipos, operadores, condicionais, listas, loops, tuplas, dicionários,
sets e conversões. Tudo isso era **dado**. Agora vem o que organiza o código:
empacotar lógica em funções, construir coleções de forma idiomática e lidar com o
que dá errado.

> Este é o notebook com **todos** os exercícios resolvidos, os rápidos e o para casa.
> Quase sempre há mais de uma resposta certa; a daqui é a que eu escreveria.

---
## 1. Funções

`def`, dois-pontos, corpo indentado. Sem declaração de tipo obrigatória e sem
chaves: a indentação delimita o corpo, como em todo o resto.

In [ ]:
def calcular_total(valor, frete):
    return valor + frete

print(calcular_total(100, 20))

120


In [ ]:
# Sem return explícito, a função devolve None
def cumprimentar(nome):
    print(f"Olá, {nome}!")

retorno = cumprimentar("Ana")
print(retorno)

Olá, Ana!
None


### Valores padrão e argumentos nomeados

Parâmetros com padrão vêm **depois** dos obrigatórios. Na chamada, você pode nomear
os argumentos: fica mais legível e libera você da ordem.

In [ ]:
def calcular_total(valor, frete=0.0, desconto=0.0):
    return valor + frete - desconto

print(calcular_total(100))                            # só o obrigatório
print(calcular_total(100, 25))                        # posicional
print(calcular_total(100, desconto=10))               # nomeado, pulando o frete
print(calcular_total(desconto=10, valor=100))         # ordem não importa se nomear

100.0
125.0
90.0
90.0


In [ ]:
# Isso levanta SyntaxError. Rode e leia a mensagem
def f(a=1, b):
    return a + b

SyntaxError: parameter without a default follows parameter with a default (3756105692.py, line 2)

### Retorno múltiplo

Python devolve uma tupla e você desempacota. É o mesmo unpacking da aula passada.

In [ ]:
def estatisticas(valores):
    return min(valores), max(valores), sum(valores) / len(valores)

menor, maior, media = estatisticas([10, 20, 30, 40])
print(menor, maior, media)

# ou receba a tupla inteira
resultado = estatisticas([1, 2, 3])
print(resultado, type(resultado))

### Docstring e type hints

Type hints são **opcionais** e não são verificados em tempo de execução: servem para
documentar e para a IDE te ajudar. Em código de projeto, use.

In [ ]:
def aplicar_desconto(valor: float, percentual: float = 0.1) -> float:
    """Retorna o valor com o desconto percentual aplicado."""
    return valor * (1 - percentual)

print(aplicar_desconto(200))
print(aplicar_desconto(200, 0.25))
print(aplicar_desconto.__doc__)

In [ ]:
# O hint não é verificado: isto roda normalmente e devolve uma string
def dobrar(x: int) -> int:
    return x * 2

print(dobrar(21))
print(dobrar("ab"))     # o hint diz int, mas ninguém confere

### Escopo

Variável criada dentro da função não existe fora. E atribuir a um nome dentro da
função cria uma variável **local**, mesmo que exista uma global com o mesmo nome.

In [ ]:
def f():
    interna = 42
    return interna

print(f())
# print(interna)   # descomente: NameError

In [ ]:
contador = 0

def incrementar():
    contador = 10      # cria uma variável local, não toca na global
    return contador

print(incrementar())
print(contador)        # continua 0

> ⚠️ **Pegadinha clássica:** nunca use lista ou dicionário como valor padrão.
> O padrão é avaliado **uma única vez**, quando a função é definida, e fica
> compartilhado entre todas as chamadas.

In [ ]:
def adicionar(item, lista=[]):     # ERRADO
    lista.append(item)
    return lista

print(adicionar("a"))
print(adicionar("b"))              # esperava ['b'], veio ['a', 'b']

In [ ]:
def adicionar(item, lista=None):   # CERTO
    if lista is None:
        lista = []
    lista.append(item)
    return lista

print(adicionar("a"))
print(adicionar("b"))

**✏️ Exercícios rápidos**

1. `media(valores)` devolve a média da lista, e `0.0` quando a lista está vazia.
2. `formatar_moeda(valor, simbolo="R$")` devolve `"R$ 1234.50"`, com 2 casas.
3. `min_max(valores)` devolve o menor **e** o maior numa só chamada.
4. `contar_status(vendas)` recebe a lista de dicionários e devolve
   `{status: quantidade}`.
5. Sem rodar, preveja a saída de `misterio()` chamada três vezes. Depois confira.

In [ ]:
vendas = [
    {"vendedor": "Ana",   "valor": 3200.0, "status": "aprovada"},
    {"vendedor": "Bruno", "valor": 150.0,  "status": "aprovada"},
    {"vendedor": "Ana",   "valor": 890.0,  "status": "cancelada"},
]
x = [1,2,3,4,5]
def misterio(x, acc={}):
    acc[x] = acc.get(x, 0) + 1
    return acc


# 1)
def media(valores: list[float]) -> float:
    """Média da lista, ou 0.0 quando ela está vazia."""
    if not valores:
        return 0.0
    return sum(valores) / len(valores)


# 2)
def formatar_moeda(valor: float, simbolo: str = "R$") -> str:
    """Formata o valor com duas casas, precedido do símbolo."""
    return f"{simbolo} {valor:.2f}"


# 3)
def min_max(valores: list[float]) -> tuple[float, float]:
    """Menor e maior da lista, numa só chamada."""
    return min(valores), max(valores)


# 4)
def contar_status(vendas: list[dict]) -> dict[str, int]:
    """Quantas vendas há em cada status."""
    contagem = {}
    for venda in vendas:
        status = vendas["status"]
        if status in contagem:
          contagem[status] + =1
        else:
          contagem[status] = 1

        print(contagem[status])
    return contagem


print(media([10, 20, 30]), "|", media([]))
print(formatar_moeda(1234.5), "|", formatar_moeda(1234.5, "US$"))
print(min_max([10, 4, 88, 7]))
print(contar_status(vendas))

In [ ]:
# 5) previsão: {'a': 1}, depois {'a': 1, 'b': 1}, depois {'a': 2, 'b': 1}
#    O dicionário padrão é criado UMA vez, quando a função é definida, e
#    sobrevive entre as chamadas. É a pegadinha da célula lá de cima.
print(misterio("a"))
print(misterio("b"))
print(misterio("a"))

# Um print só com duas chamadas mostra o mesmo dicionário duas vezes: as duas
# são avaliadas antes de imprimir, e o objeto devolvido é sempre o mesmo.
print(misterio("c"), misterio("d"))

---
## 2. Funções são objetos

Uma função pode ser guardada numa variável, passada como argumento e devolvida por
outra função. Isso destrava três ferramentas que você vai usar o tempo todo.

In [ ]:
def dobrar(x):
    return x * 2

f = dobrar              # sem parênteses: a função, não o resultado
print(f(21))
print(type(dobrar))

### `lambda`: função anônima de uma expressão

Serve para funções descartáveis, passadas como argumento. Se precisar de mais de uma
linha ou for reutilizar, escreva um `def` com nome.

In [ ]:
dobrar = lambda x: x * 2
print(dobrar(21))

soma = lambda a, b: a + b
print(soma(3, 4))

### `sorted(key=...)`

`key` recebe uma função aplicada a cada elemento para decidir a ordem. É a forma
padrão de ordenar por um critério.

In [ ]:
palavras = ["banana", "kiwi", "abacaxi", "uva"]

print(sorted(palavras))                      # alfabética
print(sorted(palavras, key=len))             # por tamanho
print(sorted(palavras, key=len, reverse=True))
print(sorted(palavras, reverse=True))

['abacaxi', 'banana', 'kiwi', 'uva']
['uva', 'kiwi', 'banana', 'abacaxi']
['abacaxi', 'banana', 'kiwi', 'uva']
['uva', 'kiwi', 'banana', 'abacaxi']


In [ ]:
# ordenando dicionários por um campo
vendas = [
    {"vendedor": "Ana",   "valor": 3200.0},
    {"vendedor": "Bruno", "valor": 150.0},
    {"vendedor": "Carla", "valor": 890.0},
]

for v in sorted(vendas, key=lambda v: v["valor"], reverse=True):
    print(v["vendedor"], v["valor"])

Ana 3200.0
Carla 890.0
Bruno 150.0


In [ ]:
# ordenando um dict pelos valores
totais = {"Ana": 4090.0, "Bruno": 150.0, "Carla": 890.0}

print(sorted(totais.items(), key=lambda par: par[0], reverse=True))

### `zip`: percorrer coleções em paralelo

Junta elemento a elemento. Para quando a menor das coleções acabar.

In [ ]:
nomes = ["Ana", "Bruno", "Carla"]
notas = [4.5, 3.0, 5.0]

for nome, nota in zip(nomes, notas):
    print(f"{nome}: {nota}")

print(dict(zip(nomes, notas)))

Ana: 4.5
Bruno: 3.0
Carla: 5.0
{'Ana': 4.5, 'Bruno': 3.0, 'Carla': 5.0}


**✏️ Exercícios rápidos**

1. Ordene `produtos` pelo preço, do mais caro para o mais barato.
2. Ordene `produtos` pelo nome, **ignorando maiúsculas/minúsculas**.
3. Com `zip`, monte `{produto: estoque}`.
4. Descubra qual vendedor tem o maior total em `totais`, usando `max` com `key`.

In [ ]:
produtos = [("Monitor", 900.0), ("teclado", 120.0), ("Mouse", 80.0)]
estoques = [4, 12, 30]
totais = {"Ana": 4090.0, "Bruno": 150.0, "Carla": 890.0}

# 1)
print(sorted(produtos, key=lambda p: p[1], reverse=True))

# 2) a key normaliza o caso só para comparar; o nome original sai intacto
print(sorted(produtos, key=lambda p: p[0].lower()))

# 3)
print(dict(zip([nome for nome, _ in produtos], estoques)))

# 4) iterar um dict dá as chaves, e o .get vira a key da comparação
print(max(totais, key=totais.get))

#    com .items() sai o par inteiro, se você quiser o total junto
print(max(totais.items(), key=lambda par: par[1]))

---
## 3. Comprehensions

Forma idiomática de **construir uma coleção** a partir de outra. Os dois blocos
abaixo fazem exatamente a mesma coisa.

In [ ]:
nums = [1, 2, 3, 4, 5, 6]

dobros = []
for n in nums:
    dobros.append(n * 2)
print(dobros)

dobros = [n * 2 for n in nums]
print(dobros)

[2, 4, 6, 8, 10, 12]
[2, 4, 6, 8, 10, 12]


A estrutura é sempre:

```
[ expressão   for item in iterável   if condição ]
     ↑              ↑                     ↑
  o que sai      de onde vem         filtro (opcional)
```

In [ ]:
nums = [1, 2, 3, 4, 5, 6]

print([n for n in nums if n % 2 == 0])          # só filtra
print([n ** 2 for n in nums])                   # só transforma
print([n ** 2 for n in nums if n % 2 != 0])     # filtra e transforma

In [ ]:
# Sobre qualquer iterável, não só listas
vendas = [
    {"vendedor": "Ana",   "valor": 3200.0, "status": "aprovada"},
    {"vendedor": "Bruno", "valor": 150.0,  "status": "aprovada"},
    {"vendedor": "Carla", "valor": 890.0,  "status": "cancelada"},
]

print([v["vendedor"] for v in vendas if v["status"] == "aprovada"])
print(sum(v["valor"] for v in vendas if v["status"] == "aprovada"))

### Dict e set comprehension

Mesma sintaxe, mudando os delimitadores. Com `:` vira dict; sem `:`, dentro de
chaves, vira set.

In [ ]:
nomes = ["Ana", "Bruno", "Carla"]

print({nome: len(nome) for nome in nomes})              # dict

original = {"a": 1, "b": 2}
print({valor: chave for chave, valor in original.items()})              # invertendo um dict

palavras = ["gato", "cachorro", "gato", "peixe"]
print({p[0] for p in palavras})                         # set: iniciais distintas

{'Ana': 3, 'Bruno': 5, 'Carla': 5}
{1: 'a', 2: 'b'}
{'c', 'g', 'p'}


### `if/else` na expressão

Quando o `if` vem **antes** do `for`, é o ternário: ele escolhe o valor, não filtra.

In [ ]:
nums = [1, 2, 3, 4]

print([n if n % 2 == 0 else 0 for n in nums])   # substitui  -> [0, 2, 0, 4]
print([n for n in nums if n % 2 == 0])          # filtra     -> [2, 4]

### Dois `for`: achatando listas aninhadas

Leia na mesma ordem em que escreveria os loops encaixados.

In [ ]:
matriz = [[1, 2], [3, 4], [5, 6]]

print([n for linha in matriz for n in linha])

> ⚠️ Comprehension serve para **construir uma coleção**. Se você só quer um efeito
> colateral (imprimir, salvar em disco), use `for` normal. E se a comprehension não
> couber confortavelmente em uma ou duas linhas, volte para o loop: legibilidade
> ganha de concisão.

**✏️ Exercícios rápidos**

1. Dos `nums`, monte a lista dos quadrados dos números maiores que 100.
2. Monte `{palavra: tamanho}` para as palavras com mais de 4 letras.
3. Monte o set das categorias distintas de `vendas`, usando `"não informada"`
   quando a chave faltar.
4. Achate `matriz` numa lista só e devolva-a ordenada, sem repetição.
5. Reescreva o loop da última célula como uma única comprehension.

In [ ]:
nums = [4, 8, 15, 16, 23, 42]

lista2 = []

for x in nums:
  if x * x > 100:
    lista2.append(x)

print([
    x for x in nums if x * x > 100
])

print(lista2)

[15, 16, 23, 42]
[15, 16, 23, 42]


In [ ]:
nums = [4, 8, 15, 16, 23, 42]
palavras = ["sol", "python", "mar", "dados", "ia"]
matriz = [[3, 1], [4, 1], [5, 9, 3]]
vendas = [
    {"vendedor": "Ana",   "valor": 3200.0, "categoria": "eletrônicos"},
    {"vendedor": "Bruno", "valor": 150.0},
    {"vendedor": "Carla", "valor": 890.0,  "categoria": "móveis"},
]

# loop do item 5
nomes_caros = []
for v in vendas:
    if v["valor"] > 500:
        nomes_caros.append(v["vendedor"].upper())
print(nomes_caros)

# 1)
print([n ** 2 for n in nums if n > 10])

# 2)
print({p: len(p) for p in palavras if len(p) > 4})

# 3) o `or` resolve os dois casos de uma vez: chave ausente, em que o .get
#    devolve None, e chave presente valendo None, que é o caso do Lab
print({v.get("categoria") or "não informada" for v in vendas})

# 4) o set tira a repetição, o sorted devolve já ordenado
print(sorted({n for linha in matriz for n in linha}))

# 5)
print([v["vendedor"].upper() for v in vendas if v["valor"] > 500])

---
## 4. try / except / finally

Em Python, exceção é mecanismo normal de fluxo, não último recurso. A cultura é
**EAFP** (*easier to ask forgiveness than permission*): tente fazer e trate o erro,
em vez de checar todas as condições antes.

In [ ]:
try:
    valor = float("abc")
except ValueError:
    print("não deu para converter")

In [ ]:
# LBYL (checar antes)  x  EAFP (tentar e tratar)
dados = {"valor": "150.0"}

# LBYL
if "valor" in dados and dados["valor"].replace(".", "").isdigit():
    print(float(dados["valor"]))

# EAFP: mais curto e cobre casos que você não previu
try:
    print(float(dados["valor"]))
except (KeyError, ValueError, TypeError):
    print("valor ausente ou inválido")

### Capture a exceção específica

`except Exception:` engole bug de verdade e transforma erro de programação em
comportamento silencioso. Capture o que você sabe tratar.

In [ ]:
def para_float(valor):
    """Converte para float, aceitando vírgula decimal. Devolve None se não der."""
    try:
        return float(str(valor).replace(",", "."))
    except (ValueError, TypeError):
        return None

for v in [15.5, "89.90", "45,50", None, "n/a"]:
    print(repr(v), "->", para_float(v))

### Exceções que você vai encontrar sempre

| Exceção | Quando |
|---|---|
| `ValueError` | tipo certo, valor inválido: `int("abc")` |
| `TypeError` | tipo errado: `"1" + 1` |
| `KeyError` | chave inexistente no dict |
| `IndexError` | índice fora da lista |
| `ZeroDivisionError` | divisão por zero |
| `FileNotFoundError` | arquivo não existe |

In [ ]:
pedido = {"id": 1, "valor": 150.0}
nums = [1, 2, 3]

for acao in ["chave", "indice", "divisao"]:
    try:
        if acao == "chave":
            pedido["categoria"]
        elif acao == "indice":
            nums[10]
        else:
            1 / 0
    except KeyError as e:
        print("KeyError:", e)
    except IndexError as e:
        print("IndexError:", e)
    except ZeroDivisionError as e:
        print("ZeroDivisionError:", e)

### `else` e `finally`

`else` roda quando **não** houve exceção. `finally` roda sempre, com ou sem erro,
com ou sem `return`. É onde vai a limpeza (fechar arquivo, conexão).

In [ ]:
def dividir(a, b):
    try:
        resultado = a / b
    except ZeroDivisionError:
        print("  divisão por zero")
        return None
    else:
        print("  deu certo")
        return resultado
    finally:
        print("  finally sempre roda")

print(dividir(10, 2))
print(dividir(10, 0))

### Levantando exceções

Quando a sua função recebe algo inválido, o certo é falhar alto, e não devolver um
valor esquisito que vai explodir três camadas adiante.

In [ ]:
def aplicar_desconto(valor: float, percentual: float) -> float:
    """Aplica um desconto entre 0 e 1 ao valor."""
    if not 0 <= percentual <= 1:
        raise ValueError(f"percentual fora do intervalo: {percentual}")
    return valor * (1 - percentual)

print(aplicar_desconto(100, 0.2))

try:
    aplicar_desconto(100, 1.5)
except ValueError as e:
    print("erro capturado:", e)

**✏️ Exercícios rápidos**

1. `parse_int(texto)` devolve o inteiro ou `None`, sem quebrar para `"abc"`, `None`
   ou `"12.5"`.
2. `pegar(lista, i, padrao=None)` devolve o elemento ou o padrão quando o índice
   não existe.
3. `soma_valores(vendas)` soma o campo `valor` **pulando** os registros em que ele
   está ausente ou é inválido, e devolve também quantos foram pulados.
4. `validar_venda(venda)` levanta `ValueError` se o valor for negativo e `KeyError`
   se faltar a chave `vendedor`. Teste os dois casos.

In [ ]:
vendas = [
    {"vendedor": "Ana",   "valor": 3200.0},
    {"vendedor": "Bruno", "valor": "150,50"},
    {"vendedor": "Carla"},
    {"vendedor": "Diego", "valor": "n/a"},
    {"vendedor": "Elena", "valor": None},
]


# 1)
def parse_int(texto) -> int | None:
    """Converte para int, devolvendo None quando não dá."""
    try:
        return int(texto)
    except (ValueError, TypeError):     # TypeError é o que o None levanta
        return None


# 2)
def pegar(lista: list, i: int, padrao=None):
    """Elemento da posição i, ou o padrão quando o índice não existe."""
    try:
        return lista[i]
    except IndexError:
        return padrao


# 3)
def soma_valores(vendas: list[dict]) -> tuple[float, int]:
    """Soma o campo `valor`, devolvendo (total, quantos foram pulados)."""
    total, pulados = 0.0, 0
    for venda in vendas:
        try:
            total += float(str(venda["valor"]).replace(",", "."))
        except (KeyError, ValueError, TypeError):
            pulados += 1
    return total, pulados


# 4)
def validar_venda(venda: dict) -> None:
    """Levanta KeyError sem vendedor e ValueError com valor negativo."""
    if "vendedor" not in venda:
        raise KeyError("vendedor")
    valor = venda.get("valor")
    if isinstance(valor, (int, float)) and valor < 0:
        raise ValueError(f"valor negativo: {valor}")


for t in ["42", "abc", None, "12.5"]:
    print(repr(t), "->", parse_int(t))

print(pegar([1, 2, 3], 1), "|", pegar([1, 2, 3], 10), "|", pegar([1, 2, 3], 10, "vazio"))
print(soma_valores(vendas))

for caso in [{"vendedor": "Ana", "valor": -79.9}, {"valor": 10.0}]:
    try:
        validar_venda(caso)
    except (KeyError, ValueError) as e:
        print(f"{type(e).__name__}: {e}")

---
# 🏠 Para casa

## Parte A: Lab do notebook 1

Volte ao Lab de relatório de vendas, no fim do notebook da aula 1, e resolva as partes 1 a 7.

Agora com uma exigência a mais: **cada item deve virar uma função**, com nome claro,
docstring e type hints. Use comprehension onde ela deixar o código mais legível, e
`for` normal onde não deixar.

Repare no que muda ao virar função: os itens deixam de depender da ordem das células,
e vários deles **encolhem para uma chamada só** porque a função aceita o campo como
argumento. `contar_por` serve para status, mês e vendedor; `total_por`, para vendedor,
mês e categoria.

In [ ]:
vendas = [
    {"vendedor": "Ana",   "produto": "notebook",     "categoria": "eletrônicos", "valor": 3200.00, "mes": "jan", "status": "aprovada"},
    {"vendedor": "Ana",   "produto": "cadeira",      "categoria": "móveis",      "valor": 890.00,  "mes": "jan", "status": "aprovada"},
    {"vendedor": "Ana",   "produto": "livro python", "categoria": "livros",      "valor": 120.00,  "mes": "fev", "status": "aprovada"},
    {"vendedor": "Ana",   "produto": "camiseta",     "categoria": "roupas",      "valor": 79.90,   "mes": "mar", "status": "aprovada"},
    {"vendedor": "Ana",   "produto": "monitor",      "categoria": "eletrônicos", "valor": 1500.00, "mes": "mar", "status": "cancelada"},
    {"vendedor": "Bruno", "produto": "mouse",        "categoria": "eletrônicos", "valor": 150.00,  "mes": "jan", "status": "aprovada"},
    {"vendedor": "Bruno", "produto": "mesa",         "categoria": "móveis",      "valor": 1200.00, "mes": "fev", "status": "aprovada"},
    {"vendedor": "Bruno", "produto": "livro python", "categoria": "livros",      "valor": 120.00,  "mes": "fev", "status": "aprovada"},
    {"vendedor": "Bruno", "produto": "fone",                                     "valor": 250.00,  "mes": "fev", "status": "aprovada"},
    {"vendedor": "Bruno", "produto": "notebook",     "categoria": "eletrônicos", "valor": 3100.00, "mes": "mar", "status": "cancelada"},
    {"vendedor": "Carla", "produto": "cadeira",      "categoria": "móveis",      "valor": 890.00,  "mes": "jan", "status": "aprovada"},
    {"vendedor": "Carla", "produto": "tênis",        "categoria": "roupas",      "valor": 320.00,  "mes": "fev", "status": "aprovada"},
    {"vendedor": "Carla", "produto": "monitor",      "categoria": "eletrônicos", "valor": 1450.00, "mes": "mar", "status": "aprovada"},
    {"vendedor": "Carla", "produto": "mesa",         "categoria": "móveis",      "valor": 1200.00, "mes": "mar", "status": "cancelada"},
    {"vendedor": "Diego", "produto": "livro sql",    "categoria": "livros",      "valor": 95.00,   "mes": "jan", "status": "aprovada"},
    {"vendedor": "Diego", "produto": "camiseta",     "categoria": "roupas",      "valor": 79.90,   "mes": "fev", "status": "aprovada"},
    {"vendedor": "Diego", "produto": "mouse",        "categoria": "eletrônicos", "valor": 150.00,  "mes": "mar", "status": "aprovada"},
    {"vendedor": "Diego", "produto": "caderno",      "categoria": None,          "valor": 35.00,   "mes": "mar", "status": "aprovada"},
]

print(len(vendas), "vendas carregadas")

### Parte 1: Explorando

In [ ]:
def categoria_de(venda: dict, padrao: str = "não informada") -> str:
    """A categoria da venda, cobrindo chave ausente e valor None de uma vez."""
    return venda.get("categoria") or padrao


def contar_por(vendas: list[dict], campo: str) -> dict[str, int]:
    """Quantas vendas há para cada valor do campo."""
    contagem = {}
    for venda in vendas:
        chave = venda.get(campo)
        contagem[chave] = contagem.get(chave, 0) + 1
    return contagem


def distintos(vendas: list[dict], campo: str) -> list[str]:
    """Os valores distintos do campo, em ordem alfabética."""
    return sorted({venda[campo] for venda in vendas})


def categorias_distintas(vendas: list[dict]) -> list[str]:
    """As categorias distintas, com as ausentes viradas em 'não informada'."""
    return sorted({categoria_de(venda) for venda in vendas})


print("1.", len(vendas), "vendas |", contar_por(vendas, "status"))
print("2.", distintos(vendas, "vendedor"))
print("3.", categorias_distintas(vendas))
print("4.", len(distintos(vendas, "produto")), "produtos distintos")

### Parte 2: Somando

In [ ]:
def aprovadas(vendas: list[dict]) -> list[dict]:
    """Só as vendas com status aprovada."""
    return [v for v in vendas if v["status"] == "aprovada"]


def faturamento(vendas: list[dict]) -> float:
    """Soma dos valores das vendas recebidas."""
    return sum(v["valor"] for v in vendas)


def ticket_medio(vendas: list[dict]) -> float:
    """Valor médio por venda, ou 0.0 se não houver nenhuma."""
    if not vendas:
        return 0.0
    return faturamento(vendas) / len(vendas)


def total_por(vendas: list[dict], campo: str) -> dict[str, float]:
    """Soma dos valores agrupada por um campo: vendedor, mês, categoria."""
    totais = {}
    for venda in vendas:
        chave = venda[campo]
        totais[chave] = totais.get(chave, 0.0) + venda["valor"]
    return totais


ok = aprovadas(vendas)
total_por_vendedor = total_por(ok, "vendedor")
por_mes = total_por(ok, "mes")

print("5.", f"faturamento {faturamento(ok):.2f} | ticket médio {ticket_medio(ok):.2f}")
print("6.", total_por_vendedor)
print("7.", por_mes, "| campeão:", max(por_mes, key=por_mes.get))

### Parte 3: Comissão por faixa

In [ ]:
def faixa_comissao(valor: float) -> float:
    """O percentual da faixa em que o valor cai.

    Sem `else`: quem chega na segunda linha já não passou na primeira.
    """
    if valor <= 1000:
        return 0.05
    if valor <= 3000:
        return 0.08
    return 0.10


def comissao_por_vendedor(vendas: list[dict]) -> dict[str, float]:
    """Comissão total de cada vendedor."""
    comissoes = {}
    for venda in vendas:
        nome = venda["vendedor"]
        ganho = venda["valor"] * faixa_comissao(venda["valor"])
        comissoes[nome] = comissoes.get(nome, 0.0) + ganho
    return comissoes


comissoes = comissao_por_vendedor(ok)

print("8.", {nome: round(v, 2) for nome, v in comissoes.items()})
print("9.", [nome for nome, valor in comissoes.items() if valor > 200])

### Parte 4: Ranking

In [ ]:
def ranking(totais: dict[str, float]) -> list[tuple[float, str]]:
    """Pares (total, nome) do maior para o menor.

    Com o total na frente, o `sorted` padrão já ordena pelo que interessa e
    usa o nome só para desempatar.
    """
    return sorted(((total, nome) for nome, total in totais.items()), reverse=True)


def imprimir_podio(totais: dict[str, float], n: int = 3) -> None:
    """Imprime os n primeiros colocados, alinhados em colunas."""
    for posicao, (total, nome) in enumerate(ranking(totais)[:n], start=1):
        print(f"{posicao}º {nome:<10} R$ {total:>10.2f}")


print("10.", ranking(total_por_vendedor))
print("11.")
imprimir_podio(total_por_vendedor)

### Parte 5: Cruzando com sets

In [ ]:
def produtos_de(vendas: list[dict], vendedor: str) -> set[str]:
    """Os produtos que um vendedor vendeu."""
    return {v["produto"] for v in vendas if v["vendedor"] == vendedor}


def conjuntos_por_vendedor(vendas: list[dict], campo: str) -> dict[str, set]:
    """Para cada vendedor, o conjunto de valores que ele tocou naquele campo."""
    mapa = {}
    for venda in vendas:
        mapa.setdefault(venda["vendedor"], set()).add(venda[campo])
    return mapa


def vendeu_em_todas_categorias(vendas: list[dict]) -> list[str]:
    """Quem vendeu em todas as categorias informadas.

    As vendas sem categoria ficam de fora dos dois lados da conta, senão
    ninguém bateria com o conjunto completo.
    """
    com_categoria = [v for v in vendas if v.get("categoria")]
    informadas = {v["categoria"] for v in com_categoria}
    por_vendedor = conjuntos_por_vendedor(com_categoria, "categoria")
    return sorted(nome for nome, cats in por_vendedor.items() if cats == informadas)


def produto_de_todos(vendas: list[dict]) -> set[str]:
    """Os produtos vendidos por todo mundo. A interseção pode dar vazia."""
    por_vendedor = conjuntos_por_vendedor(vendas, "produto")
    return set.intersection(*por_vendedor.values())


ana, bruno = produtos_de(vendas, "Ana"), produtos_de(vendas, "Bruno")

print("12. só a Ana:", ana - bruno)
print("    os dois :", ana & bruno)
print("13.", vendeu_em_todas_categorias(vendas) or "ninguém")
print("14.", produto_de_todos(vendas) or "nenhum")

### Parte 6: Relatório final

In [ ]:
def imprimir_relatorio(vendas: list[dict]) -> None:
    """Uma linha por vendedor: vendas, total e percentual do faturamento."""
    totais = total_por(vendas, "vendedor")
    quantidades = contar_por(vendas, "vendedor")
    geral = sum(totais.values())

    print(f"{'Vendedor':<10} {'Vendas':>7} {'Total':>12} {'%':>7}")
    for total, nome in ranking(totais):
        print(f"{nome:<10} {quantidades[nome]:>7} {total:>12.2f} {total / geral:>7.1%}")


print("15.")
imprimir_relatorio(ok)

15.


NameError: name 'ok' is not defined

### Parte 7: Desafios

In [ ]:
def produtos_em_ordem(vendas: list[dict]) -> list[str]:
    """Os produtos na ordem da primeira aparição, sem repetir.

    O set responde rápido se já viu; a lista é quem guarda a ordem. Um set
    sozinho não resolve, porque ele não tem ordem nenhuma.
    """
    vistos, ordem = set(), []
    for venda in vendas:
        if venda["produto"] not in vistos:
            vistos.add(venda["produto"])
            ordem.append(venda["produto"])
    return ordem


def caras_mesmo_com_desconto(vendas: list[dict], minimo: float = 1000.0,
                             desconto: float = 0.1) -> None:
    """Imprime as vendas que passam do mínimo mesmo depois do desconto."""
    for venda in vendas:
        if (com_desconto := venda["valor"] * (1 - desconto)) > minimo:
            print(f"  {venda['vendedor']:<6} {venda['produto']:<13}"
                  f" {venda['valor']:>9.2f} -> {com_desconto:>9.2f}")


print("16.", produtos_em_ordem(vendas))
print("17.")
caras_mesmo_com_desconto(ok)

## Parte B: pipeline de limpeza

Os dados abaixo são o mesmo relatório de vendas, só que como chegam na vida real:
valor em formatos inconsistentes, chave faltando, `None`, texto solto, status escrito
de jeitos diferentes.

Este é, em miniatura, exatamente o trabalho que vamos fazer com Pandas mais para a
frente. Vale sentir a dor manualmente antes.

In [ ]:
vendas_sujas = [
    {"vendedor": "Ana",    "produto": "notebook", "categoria": "eletrônicos", "valor": 3200.00,   "mes": "jan", "status": "aprovada"},
    {"vendedor": "ana",    "produto": "cadeira",  "categoria": "móveis",      "valor": "890,00",  "mes": "JAN", "status": "Aprovada"},
    {"vendedor": "Bruno",  "produto": "mouse",    "categoria": None,          "valor": "150.00",  "mes": "jan", "status": "aprovada"},
    {"vendedor": "Bruno",  "produto": "mesa",     "categoria": "móveis",      "valor": None,      "mes": "fev", "status": "APROVADA"},
    {"vendedor": " Carla", "produto": "fone",                                 "valor": "R$ 250",  "mes": "fev", "status": "cancelada"},
    {"vendedor": "Carla",  "produto": "monitor",  "categoria": "eletrônicos", "valor": "n/a",     "mes": "fev", "status": "aprovada"},
    {"vendedor": "Diego",  "produto": "livro",    "categoria": "livros",      "valor": 120.00,    "mes": "mar", "status": "pendente"},
    {"vendedor": "Diego",  "produto": "camiseta", "categoria": "roupas",      "valor": -79.90,    "mes": "mar", "status": "aprovada"},
    {"vendedor": "Elena",  "produto": "notebook", "categoria": "eletrônicos", "valor": "3.100,00","mes": "mar", "status": "aprovada"},
]

print(f"{len(vendas_sujas)} registros")

**1.** `normalizar_texto(valor, padrao="não informada")` devolve o texto sem espaços
nas pontas e em minúsculas. Trata `None` e chave ausente devolvendo o padrão.

In [ ]:
def normalizar_texto(valor, padrao: str = "não informada") -> str:
    """Texto sem espaços nas pontas e em minúsculas.

    O `not valor` cobre de uma vez a chave ausente, que chega como None vinda
    do .get, o None gravado no próprio campo e a string vazia.
    """
    if not valor:
        return padrao
    return str(valor).strip().lower()


for v in ["  Ana ", "MÓVEIS", None, ""]:
    print(repr(v), "->", repr(normalizar_texto(v)))

**2.** `normalizar_valor(valor)` devolve `float` ou `None`. Precisa dar conta de:
número, `"890,00"`, `"150.00"`, `"3.100,00"` (milhar com ponto), `"R$ 250"`, `"n/a"`
e `None`.

_Dica: limpe a string antes de converter e use `try/except`._

In [ ]:
def normalizar_valor(valor) -> float | None:
    """Converte para float, devolvendo None quando não dá.

    Dá conta de número, "890,00", "150.00", "3.100,00" (milhar com ponto),
    "R$ 250", "n/a" e None.
    """
    if isinstance(valor, (int, float)) and not isinstance(valor, bool):
        return float(valor)

    # joga fora tudo o que não é dígito nem separador: "R$", espaço, letra
    texto = "".join(c for c in str(valor) if c.isdigit() or c in ".,-")

    if "," in texto and "." in texto:      # 3.100,00 -> o ponto é milhar
        texto = texto.replace(".", "").replace(",", ".")
    elif "," in texto:                     # 890,00
        texto = texto.replace(",", ".")

    try:
        return float(texto)
    except ValueError:                     # sobrou "" ou algo sem número
        return None


for v in [3200.0, "890,00", "150.00", "3.100,00", "R$ 250", "n/a", None, -79.90]:
    print(f"{str(v):<12} -> {normalizar_valor(v)}")

**3.** `limpar(vendas)` devolve **uma nova lista** (sem alterar a original) em que
cada registro tem `vendedor`, `produto`, `categoria`, `mes` e `status` normalizados,
e `valor` já como float. Descarte registros com valor inválido, ausente ou negativo.

Devolva também a lista dos registros descartados, para poder auditar o que se perdeu.

In [ ]:
TEXTUAIS = ("vendedor", "produto", "categoria", "mes", "status")


def limpar(vendas: list[dict]) -> tuple[list[dict], list[dict]]:
    """Normaliza os registros e separa os que não dá para aproveitar.

    Devolve (limpas, descartadas). A lista original não é tocada: cada
    registro limpo é um dicionário novo, montado por comprehension.
    """
    limpas, descartadas = [], []
    for venda in vendas:
        valor = normalizar_valor(venda.get("valor"))
        if valor is None or valor < 0:
            descartadas.append(venda)
            continue
        novo = {campo: normalizar_texto(venda.get(campo)) for campo in TEXTUAIS}
        novo["valor"] = valor
        limpas.append(novo)
    return limpas, descartadas


limpas, descartadas = limpar(vendas_sujas)

print(f"{len(limpas)} aproveitadas, {len(descartadas)} descartadas\n")
for v in limpas:
    print(" ", v)

print("\ndescartadas:", [(v["vendedor"], v.get("valor")) for v in descartadas])
print("original intacta:", repr(vendas_sujas[4]["vendedor"]))

**4.** `resumo(vendas)` recebe a lista **já limpa** e devolve:

```python
{
    "total_registros": 0,
    "faturamento": 0.0,
    "ticket_medio": 0.0,
    "por_vendedor": {},
    "por_categoria": {},
    "top_vendedor": "",
}
```

Considere apenas as vendas aprovadas. Levante `ValueError` se a lista vier vazia.

In [ ]:
def total_por(vendas: list[dict], campo: str) -> dict[str, float]:
    """Soma do valor agrupada por um campo.

    É a mesma do Lab, repetida aqui para a Parte B rodar sozinha.
    """
    totais = {}
    for venda in vendas:
        totais[venda[campo]] = totais.get(venda[campo], 0.0) + venda["valor"]
    return totais


def resumo(vendas: list[dict]) -> dict:
    """Os números do relatório, considerando só as vendas aprovadas.

    Raises:
        ValueError: se a lista vier vazia.
    """
    if not vendas:
        raise ValueError("não há vendas para resumir")

    ok = [v for v in vendas if v["status"] == "aprovada"]
    por_vendedor = total_por(ok, "vendedor")
    faturado = sum(v["valor"] for v in ok)

    return {
        "total_registros": len(ok),
        "faturamento": faturado,
        "ticket_medio": faturado / len(ok) if ok else 0.0,
        "por_vendedor": por_vendedor,
        "por_categoria": total_por(ok, "categoria"),
        "top_vendedor": max(por_vendedor, key=por_vendedor.get) if por_vendedor else "",
    }


for chave, valor in resumo(limpas).items():
    print(f"{chave:<16} {valor}")

try:
    resumo([])
except ValueError as e:
    print("\nValueError:", e)

**5. Desafio:** `agrupar_por(vendas, chave)` genérica, que recebe o nome de um campo e
devolve `{valor_do_campo: [registros]}`. Depois reescreva `por_vendedor` e
`por_categoria` do item 4 usando ela.

_Isto é, essencialmente, o `groupby` do Pandas escrito à mão._

In [ ]:
def agrupar_por(vendas: list[dict], chave: str) -> dict[str, list[dict]]:
    """Agrupa os registros pelo valor de um campo.

    O `setdefault` cria a lista na primeira vez que a chave aparece, o que
    dispensa o `if chave in grupos` a cada volta.
    """
    grupos = {}
    for venda in vendas:
        grupos.setdefault(venda[chave], []).append(venda)
    return grupos


def somar(vendas: list[dict]) -> float:
    """Soma o campo valor dos registros recebidos."""
    return sum(v["valor"] for v in vendas)


def total_por(vendas: list[dict], campo: str) -> dict[str, float]:
    """A mesma do item 4, agora escrita em cima do agrupar_por."""
    return {chave: somar(grupo) for chave, grupo in agrupar_por(vendas, campo).items()}


limpas_ok = [v for v in limpas if v["status"] == "aprovada"]

print({c: len(g) for c, g in agrupar_por(limpas_ok, "categoria").items()})
print(total_por(limpas_ok, "vendedor"))

# O `resumo` do item 4 não muda uma linha e já usa a versão nova: o Python
# procura o nome `total_por` na hora da chamada, não na hora da definição.
print(resumo(limpas)["por_categoria"])

# E é isto que o pandas faz em df.groupby("vendedor")["valor"].sum():
# separar as linhas por chave, aplicar em cada grupo e juntar o resultado.

**6. Desafio:** imprima o relatório final alinhado em colunas: vendedor, número de
vendas, total e percentual do faturamento, ordenado do maior para o menor.

```
Vendedor     Vendas        Total      %
ana               2      4090.00   62.3%
bruno             1      1200.00   18.3%
```

In [ ]:
def imprimir_relatorio_final(vendas: list[dict]) -> None:
    """Vendedor, número de vendas, total e percentual, do maior para o menor."""
    numeros = resumo(vendas)
    grupos = agrupar_por([v for v in vendas if v["status"] == "aprovada"], "vendedor")

    print(f"{'Vendedor':<12} {'Vendas':>7} {'Total':>12} {'%':>7}")
    for nome, total in sorted(numeros["por_vendedor"].items(),
                              key=lambda par: par[1], reverse=True):
        print(f"{nome:<12} {len(grupos[nome]):>7} {total:>12.2f}"
              f" {total / numeros['faturamento']:>7.1%}")


imprimir_relatorio_final(limpas)

---
### Na próxima

Vamos empacotar essas funções num módulo de verdade: arquivos `.py`, imports,
ambiente e estrutura de projeto, para sair do notebook e virar código reaproveitável.